# TraceXMail Empirical Evaluation & Results Walkthrough

This notebook provides an interactive walkthrough of the empirical findings from the TraceXMail evaluation suite:
- Class balance and group-split distributions
- Benchmark model comparisons (Accuracy, Macro-F1, Brier Score, ECE)
- Layered pipeline ablation heatmap & cost-latency Pareto frontier
- Probability calibration before and after temperature scaling
- Error clusters and feature deletion faithfulness curves (AUDC)

In [1]:
import json
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load generated report files
with open('../reports/benchmark.json', 'r') as f:
    benchmark_data = json.load(f)

with open('../reports/ablation.json', 'r') as f:
    ablation_data = json.load(f)

with open('../reports/error_analysis.json', 'r') as f:
    error_data = json.load(f)

print(f"Loaded Benchmark evaluated on {benchmark_data['splits']['test_count']} clean test records.")
print(f"Best baseline identified: {benchmark_data['best_baseline']}")

## 1. Benchmark Comparison Table
Compares all baseline models on identical clean test splits.

In [2]:
df_models = pd.DataFrame([
    {
        'Model': m['name'],
        'Accuracy': f"{m['accuracy']*100:.1f}%",
        'Macro-F1': f"{m['macroF1']:.4f}",
        'Weighted-F1': f"{m['weightedF1']:.4f}",
        'Brier Score': m['brierScore'],
        'ECE': m['ece'],
        'FPR @ 95% Recall': f"{m['fprAt95Recall']*100:.1f}%"
    }
    for m in benchmark_data['models']
])
display(df_models)

## 2. Macro-F1 Comparison Plot

In [3]:
fig, ax = plt.subplots(figsize=(10, 5))
names = [m['name'] for m in benchmark_data['models']]
f1s = [m['macroF1'] for m in benchmark_data['models']]
colors = ['#94A3B8', '#38BDF8', '#818CF8', '#10B981', '#F59E0B']

bars = ax.barh(names, f1s, color=colors)
ax.set_xlabel('Macro-F1 Score')
ax.set_title('TraceXMail vs Baseline Models on Clean Test Split (Prompt 26)')
ax.set_xlim(0, 1.0)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.02, bar.get_y() + bar.get_height()/2, f'{w:.3f}', va='center')
plt.tight_layout()
plt.show()

## 3. Layered Pipeline Ablation (Cost vs F1 Trade-off)

In [4]:
df_ablation = pd.DataFrame(ablation_data['all_configurations'])
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(df_ablation['costPer1kEmails'], df_ablation['macroF1'], s=df_ablation['meanLatencyMs']*2, alpha=0.7, c='#38BDF8', edgecolors='white')

for i, row in df_ablation.iterrows():
    ax.annotate(row['configName'], (row['costPer1kEmails'], row['macroF1'] + 0.01), fontsize=8)

ax.set_xlabel('Cost per 1,000 Emails ($)')
ax.set_ylabel('Macro-F1')
ax.set_title('Pareto Frontier: Macro-F1 vs Cost per 1,000 Emails (Prompt 27)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 4. Explanation Faithfulness: Feature Deletion Curves (AUDC)
Compares model confidence degradation when deleting top-attributed features vs random features.

In [5]:
steps = error_data['deletion_faithfulness_test']['deletion_steps']
top_k = error_data['deletion_faithfulness_test']['top_k_confidence_curve']
rand = error_data['deletion_faithfulness_test']['random_feature_curve']

plt.figure(figsize=(8, 4.5))
plt.plot(steps, top_k, marker='o', label=f"Top-k Features (AUDC: {error_data['deletion_faithfulness_test']['audc_top_k']})", color='#EF4444', lw=2)
plt.plot(steps, rand, marker='s', label=f"Random Features (AUDC: {error_data['deletion_faithfulness_test']['audc_random']})", color='#94A3B8', linestyle='--', lw=2)
plt.xlabel('Number of Zeroed Features')
plt.ylabel('Model Confidence')
plt.title('Explanation Faithfulness Deletion Test (Prompt 30)')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

print(f"Faithfulness Ratio: {error_data['deletion_faithfulness_test']['faithfulness_ratio']}x faster confidence collapse on true features.")